# nnU-Net BraTS 2024 MEN-RT — Google Colab Pipeline

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run cells **in order**, top to bottom
3. Keep the browser tab open during training

**All configurable parameters are in Cell 3 only.**

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
import subprocess, torch

smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(smi.stdout)

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU found. Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Mount Google Drive (checkpoints saved permanently here) ───────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/nnunet_checkpoints'
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted. Checkpoints will be saved to: {DRIVE_DIR}')

In [ ]:
# ── Cell 3: Configure Environment ────────────────────────────────────────────
# ALL configurable parameters are here. Edit ONLY this cell.

import os, sys
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT       = '/content/nnunet'
DATA_DIR      = '/content/data/BraTS-MEN-RT-Train-v2'
DRIVE_OUTPUT  = '/content/drive/MyDrive/nnunet_checkpoints'

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_ID        = '001'
DATASET_NAME      = 'BraTSMENRT'
LABEL_SUFFIX      = 'gtv'
MAX_TRAIN_CASES   = 20

# ── Training ──────────────────────────────────────────────────────────────────
NNUNET_CONFIGURATION  = '3d_fullres'
NNUNET_TRAINER_CLASS  = 'nnUNetTrainerEarlyStopping'
NNUNET_PLANS          = 'nnUNetPlans'
NUM_FOLDS             = 5
NNUNET_SEED           = 42
NNUNET_NUM_EPOCHS     = 10
RESUME_TARGET_EPOCHS  = 50

# ── Early stopping ────────────────────────────────────────────────────────────
ES_PATIENCE   = 50
ES_MIN_DELTA  = 0.0001
ES_WARMUP     = 50

# ── Hardware ──────────────────────────────────────────────────────────────────
CUDA_VISIBLE_DEVICES  = '0'
NNUNET_N_PROC_DA      = '2'
NNUNET_COMPILE        = 'false'

# ── Inference ─────────────────────────────────────────────────────────────────
INFERENCE_STEP_SIZE   = 0.5
INFERENCE_DISABLE_TTA = 'false'
INFERENCE_DEVICE      = 'cuda'

# ── Evaluation ────────────────────────────────────────────────────────────────
PP_THRESHOLDS      = '10,25,50,100,200,500'
NSD_TOLERANCE_MM   = 2.0
LOW_DICE_THRESHOLD = 0.5
BOOTSTRAP_N        = 2000

# ─────────────────────────────────────────────────────────────────────────────
# Inject into os.environ (do not edit below)
# ─────────────────────────────────────────────────────────────────────────────
os.environ.update({
    'nnUNet_raw'             : f'{PROJECT}/nnunet_raw',
    'nnUNet_preprocessed'    : f'{PROJECT}/nnunet_preprocessed',
    'nnUNet_results'         : f'{PROJECT}/nnunet_results',
    'DATASET_ID'             : DATASET_ID,
    'DATASET_NAME'           : DATASET_NAME,
    'LABEL_SUFFIX'           : LABEL_SUFFIX,
    'MAX_TRAIN_CASES'        : str(MAX_TRAIN_CASES),
    'TRAIN_SOURCE'           : DATA_DIR,
    'VAL_SOURCE'             : '',
    'NNUNET_CONFIGURATION'   : NNUNET_CONFIGURATION,
    'NNUNET_TRAINER_CLASS'   : NNUNET_TRAINER_CLASS,
    'NNUNET_PLANS_IDENTIFIER': NNUNET_PLANS,
    'NUM_FOLDS'              : str(NUM_FOLDS),
    'NNUNET_SEED'            : str(NNUNET_SEED),
    'NNUNET_NUM_EPOCHS'      : str(NNUNET_NUM_EPOCHS),
    'RESUME_TARGET_EPOCHS'   : str(RESUME_TARGET_EPOCHS),
    'ES_PATIENCE'            : str(ES_PATIENCE),
    'ES_MIN_DELTA'           : str(ES_MIN_DELTA),
    'ES_WARMUP'              : str(ES_WARMUP),
    'INFERENCE_STEP_SIZE'    : str(INFERENCE_STEP_SIZE),
    'INFERENCE_DISABLE_TTA'  : INFERENCE_DISABLE_TTA,
    'INFERENCE_DEVICE'       : INFERENCE_DEVICE,
    'PP_THRESHOLDS'          : PP_THRESHOLDS,
    'NSD_TOLERANCE_MM'       : str(NSD_TOLERANCE_MM),
    'LOW_DICE_THRESHOLD'     : str(LOW_DICE_THRESHOLD),
    'BOOTSTRAP_N'            : str(BOOTSTRAP_N),
    'CUDA_VISIBLE_DEVICES'   : CUDA_VISIBLE_DEVICES,
    'nnUNet_n_proc_DA'       : NNUNET_N_PROC_DA,
    'nnUNet_compile'         : NNUNET_COMPILE,
    'EXPERIMENT_NAME'        : 'BraTSMENRT_baseline',
    'KAGGLE_OUTPUT_DIR'      : DRIVE_OUTPUT,
    'PYTHONUNBUFFERED'       : '1',
})

for d in ['nnunet_raw','nnunet_preprocessed','nnunet_results',
          'metrics','results','visualizations','inference_outputs','logs','checkpoints']:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

print('Environment configured:')
print(f'  Dataset       : {DATASET_NAME} (ID={DATASET_ID})')
print(f'  Train cases   : {MAX_TRAIN_CASES}')
print(f'  Folds         : {NUM_FOLDS}')
print(f'  Epochs/fold   : {NNUNET_NUM_EPOCHS}')
print(f'  Trainer       : {NNUNET_TRAINER_CLASS}')
print(f'  Data source   : {DATA_DIR}')
print(f'  Checkpoint dir: {DRIVE_OUTPUT}')

In [ ]:
# ── Cell 4: Clone Repository + Install Dependencies ───────────────────────────
import subprocess, sys
from pathlib import Path

if not Path('/content/nnunet/.git').exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/maryampervaiz-alt/nnunet-training.git', '/content/nnunet'],
        check=True
    )
    print('Repo cloned.')
else:
    subprocess.run(['git', '-C', '/content/nnunet', 'pull'], check=True)
    print('Repo already exists, pulled latest.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                '/content/nnunet/requirements.txt', '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 5: Register Custom Trainer with nnU-Net ──────────────────────────────
import shutil, importlib.util
from pathlib import Path
import nnunetv2

nnunet_pkg_dir = Path(nnunetv2.__file__).parent
trainer_src    = Path('/content/nnunet/src/training/nnunet_trainer_es.py')
trainer_dst    = nnunet_pkg_dir / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'

if not trainer_src.exists():
    raise FileNotFoundError(f'Trainer not found: {trainer_src}')

shutil.copy2(trainer_src, trainer_dst)

spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
cls  = getattr(mod, 'nnUNetTrainerEarlyStopping', None)
if cls is None:
    raise ImportError('nnUNetTrainerEarlyStopping class not found')

print(f'Custom trainer registered: {trainer_dst.name}')
print('Trainer registration: OK')

In [ ]:
# ── Cell 6: Set Working Directory ─────────────────────────────────────────────
import os, sys

os.chdir('/content/nnunet')
if '/content/nnunet' not in sys.path:
    sys.path.insert(0, '/content/nnunet')

print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── Cell 7: STEP 1 — Prepare Dataset ─────────────────────────────────────────
# Converts BraTS data to nnU-Net format. ~2-4 minutes for 20 cases.
import subprocess, sys, os
from pathlib import Path

print(f'Source : {os.environ["TRAIN_SOURCE"]}')
print(f'Cases  : {os.environ["MAX_TRAIN_CASES"]}')
print()

if not Path(os.environ['TRAIN_SOURCE']).exists():
    raise FileNotFoundError(
        f'Data not found: {os.environ["TRAIN_SOURCE"]}\n'
        'Make sure you ran the Kaggle download cell and the path is correct.'
    )

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/01_prepare_dataset.py',
     '--train',     os.environ['TRAIN_SOURCE'],
     '--skip-val',
     '--log-dir',   'logs',
     '--max-cases', os.environ['MAX_TRAIN_CASES']],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Data preparation failed (rc={proc.returncode})')
print('Dataset preparation complete.')

In [ ]:
# ── Cell 8: Verify Dataset Structure ─────────────────────────────────────────
import json
from pathlib import Path

raw_dir     = Path(os.environ['nnUNet_raw'])
dataset_dir = raw_dir / 'Dataset001_BraTSMENRT'

images = sorted((dataset_dir / 'imagesTr').glob('*.nii.gz'))
labels = sorted((dataset_dir / 'labelsTr').glob('*.nii.gz'))
print(f'Images : {len(images)}')
print(f'Labels : {len(labels)}')

ds_json = dataset_dir / 'dataset.json'
if ds_json.exists():
    d = json.loads(ds_json.read_text())
    print(f'dataset.json: {d.get("numTraining")} training cases')

if len(images) == 0:
    raise RuntimeError('No images found — check Cell 7 output for errors.')
print('Dataset structure OK.')

In [ ]:
# ── Cell 9: STEP 2 — nnU-Net Planning & Preprocessing ────────────────────────
# ~5 minutes for 20 cases.
import subprocess, sys, os

print('Running nnUNetv2_plan_and_preprocess ...')
proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Preprocessing failed (rc={proc.returncode})')
print('Preprocessing complete.')

In [ ]:
# ── Cell 10: STEP 3 — Train All Folds ────────────────────────────────────────
# Runtime: ~4-5 min/epoch x 10 epochs x 5 folds = ~3-4 hours on T4
# Checkpoints saved to Google Drive after every fold.

import os, subprocess, sys, shutil
from pathlib import Path

NUM_FOLDS    = int(os.environ['NUM_FOLDS'])
NUM_EPOCHS   = int(os.environ['NNUNET_NUM_EPOCHS'])
TRAINER      = os.environ['NNUNET_TRAINER_CLASS']
PLANS        = os.environ['NNUNET_PLANS_IDENTIFIER']
DRIVE_OUTPUT = os.environ['KAGGLE_OUTPUT_DIR']

def _ckpt_dir(fold):
    ds  = f"Dataset{os.environ['DATASET_ID'].zfill(3)}_{os.environ['DATASET_NAME']}"
    cfg = os.environ['NNUNET_CONFIGURATION']
    return (Path(os.environ['nnUNet_results']) / ds
            / f"{TRAINER}__{PLANS}__{cfg}" / f'fold_{fold}')

def _save_to_drive(fold):
    src = _ckpt_dir(fold)
    dst = (Path(DRIVE_OUTPUT) / 'nnunet_results'
           / src.relative_to(os.environ['nnUNet_results']))
    dst.mkdir(parents=True, exist_ok=True)
    for fp in list(src.glob('*.pth')) + list(src.glob('training_log*.txt')):
        shutil.copy2(fp, dst / fp.name)
    mcsv = Path('metrics') / f'fold_{fold}_training.csv'
    if mcsv.exists():
        md = Path(DRIVE_OUTPUT) / 'metrics'
        md.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mcsv, md / mcsv.name)
    print(f'  [fold {fold}] Saved to Drive: {dst}')

print(f'Training {NUM_FOLDS} folds x {NUM_EPOCHS} epochs')
fold_results = {}

for fold in range(NUM_FOLDS):
    print()
    print('=' * 64)
    print(f'  Training fold {fold}/{NUM_FOLDS-1} | {NUM_EPOCHS} epochs')
    print('=' * 64)

    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/03_train.py',
         '--folds',           str(fold),
         '--num-epochs',      str(NUM_EPOCHS),
         '--seed',            os.environ['NNUNET_SEED'],
         '--trainer',         TRAINER,
         '--es-patience',     os.environ['ES_PATIENCE'],
         '--es-min-delta',    os.environ['ES_MIN_DELTA'],
         '--es-warmup',       os.environ['ES_WARMUP'],
         '--log-dir',         'logs',
         '--metrics-dir',     'metrics',
         '--checkpoints-dir', 'checkpoints'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    fold_results[fold] = proc.returncode

    status = 'OK' if proc.returncode == 0 else f'FAILED (rc={proc.returncode})'
    print(f'Fold {fold}: {status}')
    _save_to_drive(fold)

print()
print('=' * 64)
print('Training complete:')
for fold, rc in fold_results.items():
    print(f'  Fold {fold}: {"OK" if rc == 0 else "FAILED"}')
print('=' * 64)
print(f'Checkpoints saved to: {DRIVE_OUTPUT}')

---
## After Training

All checkpoints are saved to **Google Drive → nnunet_checkpoints**.

To resume on university GPU:
```bash
kaggle datasets download maryampervaiz24029/nnunet-brats-checkpoints -p nnunet_results --unzip
python scripts/03_train.py --continue-training --folds 0 1 2 3 4 --num-epochs 50
```